# Tiền xử lý Ames Housing & phòng tránh Data Leakage

Các bước: loại cột định danh → chia Train/Val/Test theo `Yr Sold` → kiểm toán NaN trên Train → xây Pipeline (chỉ fit trên Train).
Notebook dừng ở bước chuẩn bị đặc trưng, chưa huấn luyện mô hình.

## 1. Đọc dữ liệu, loại cột định danh

- `Order`, `PID` chỉ là mã định danh từng dòng, không mô tả căn nhà → loại bỏ.
- `MS SubClass` có kiểu số nhưng thực chất là **mã loại nhà** (20, 60, 120...). Phép so sánh lớn/nhỏ hay cộng trừ trên các mã này vô nghĩa (mã 60 không phải "gấp 3 lần" mã 20) → chuyển sang chuỗi để xử lý như biến phân loại.

In [1]:
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = pd.read_csv("AmesHousing - AmesHousing.csv.csv")   # nếu dùng file Excel: pd.read_excel("AmesHousing.xlsx")
print("Kích thước ban đầu:", df.shape)

df = df.drop(columns=["Order", "PID"])
#Ép kiểu MS SubClass thành biến phân loại (categorical)
df["MS SubClass"] = df["MS SubClass"].astype(str)
print(f"Kích thước dữ liệu sau khi làm sạch cột định danh: {df.shape}")

Kích thước ban đầu: (2930, 82)
Kích thước dữ liệu sau khi làm sạch cột định danh: (2930, 80)


## 2. Chia dữ liệu theo thời gian (không shuffle)

| Tập | Năm bán | Số giao dịch |
|---|---|---:|
| Train | 2006–2008 | 1.941 |
| Validation | 2009 | 648 |
| Test | 2010 | 341 |

Dùng quá khứ để dự đoán tương lai, nên Validation/Test không được tham gia học bất kỳ tham số tiền xử lý nào. `SalePrice` được tách riêng làm nhãn `y`.

In [2]:
train_df = df[df["Yr Sold"].between(2006, 2008)]
val_df   = df[df["Yr Sold"] == 2009]
test_df  = df[df["Yr Sold"] == 2010]
target="SalePrice"
X_train, y_train = train_df.drop(columns=[target]), train_df[target]
X_val, y_val = val_df.drop(columns=[target]), val_df[target]
X_test, y_test = test_df.drop(columns=[target]), test_df[target]

# Kiểm tra số lượng dòng khớp với đề bài
print(f"Train (2006-2008): {X_train.shape} giao dịch")
print(f"Validation (2009): {X_val.shape} giao dịch")
print(f"Test (2010):       {X_test.shape} giao dịch")

Train (2006-2008): (1941, 79) giao dịch
Validation (2009): (648, 79) giao dịch
Test (2010):       (341, 79) giao dịch


## 3. Kiểm toán dữ liệu thiếu (chỉ trên Train)

In [3]:
missing = pd.DataFrame({
    "Số lượng NaN": X_train.isna().sum(),
    "Tỷ lệ (%)": (X_train.isna().mean() * 100).round(2),
})
missing = missing[missing["Số lượng NaN"] > 0].sort_values("Tỷ lệ (%)", ascending=False)
missing

,Số lượng NaN,Tỷ lệ (%)
Pool QC,1928,99.33
Misc Feature,1878,96.75
Alley,1805,92.99
Fence,1576,81.20
Mas Vnr Type,1172,60.38
Fireplace Qu,940,48.43
Lot Frontage,321,16.54
Garage Qual,107,5.51
Garage Cond,107,5.51
Garage Yr Blt,107,5.51


### Quyết định xử lý

| Nhóm | Cột | Cách xử lý |
|---|---|---|
| Structural Missing | `Pool QC`, `Alley`, `Fence`, `Fireplace Qu`, `Garage Type`, `Bsmt Qual` | NaN nghĩa là *không có tiện ích* → điền `"None"` (một category thật, không phải Python `None`) |
| `Electrical` | `Electrical` | Chỉ thiếu 1 mẫu → điền mode (`most_frequent`). Đây là phương án xử lý thực dụng, không khẳng định đã chứng minh MCAR |
| `Lot Frontage` | `Lot Frontage` | Thiếu 321/1.941 mẫu (16,54%) → điền median + `add_indicator=True` để giữ thêm tín hiệu "ô này ban đầu bị thiếu". Có thể giả thuyết MAR nhưng không kết luận cơ chế thiếu chỉ từ tỷ lệ NaN |
| Các cột còn lại | `Misc Feature`, `Mas Vnr Type`, các cột Garage/Bsmt khác... | Số: điền median. Phân loại: điền `"Missing"` (giữ trạng thái chưa biết, không khẳng định là không có tiện ích) |

Đề xuất cải thiện nếu giữ các cột còn lại: `Misc Feature` và `Mas Vnr Type` (khi `Mas Vnr Area = 0`) có thể điền `"None"`; các cột Garage/Bsmt điền `"None"` khi nhà không có garage/tầng hầm (và số liên quan điền 0).

## 4. Pipeline + ColumnTransformer

Mỗi nhóm cột có một Pipeline riêng. `handle_unknown="ignore"` giúp One-Hot không lỗi khi Val/Test có category mà Train chưa thấy.
* Xây dựng `ColumnTransformer` kết hợp `Pipeline` để tự động hóa việc điền giá trị thiếu (`SimpleImputer`) và chuẩn hóa/mã hóa dữ liệu.
* **Nguyên tắc chống rò rỉ (Data Leakage):** Chỉ gọi `.fit_transform()` trên tập `X_train` để học các tham số thống kê (trung vị, trung bình, độ lệch chuẩn, danh mục). Sau đó dùng chính bộ biến đổi đã fit đó để gọi `.transform()` cho `X_val` và `X_test`.

In [5]:
structural_cols = ["Pool QC", "Alley", "Fence", "Fireplace Qu", "Garage Type", "Bsmt Qual"]
electrical_cols = ["Electrical"]
lot_cols = ["Lot Frontage"]

num_cols = [c for c in X_train.select_dtypes("number").columns if c not in lot_cols]
cat_cols = [c for c in X_train.select_dtypes("object").columns
            if c not in structural_cols + electrical_cols]

ohe = lambda: OneHotEncoder(handle_unknown="ignore", sparse_output=False)
num_pipline=Pipeline([
    ('imputer',SimpleImputer(strategy='median')),
    ('scaler',StandardScaler())
])
struct_pipline=Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="None")),
    ("enc", ohe())
])
elec_pipline=Pipeline([
    ("imp", SimpleImputer(strategy="most_frequent")), 
    ("enc", ohe())
])
lot_pipline=Pipeline([
    ("imp", SimpleImputer(strategy="median", add_indicator=True)), 
    ("sc", StandardScaler())
])
cate_pipline=Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="Missing")), 
    ("enc", ohe())
])
preprocessor = ColumnTransformer([
    ("structural", struct_pipline, structural_cols),
    ("electrical", elec_pipline, electrical_cols),
    ("lot_frontage", lot_pipline, lot_cols),
    ("numeric", num_pipline, num_cols),
    ("categorical", cate_pipline, cat_cols),
])

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_17840\934983479.py:6: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = [c for c in X_train.select_dtypes("object").columns


## 5. Fit trên Train, chỉ transform Validation/Test

Đây là bước chống Data Leakage: median, mode, mean/std và danh mục One-Hot đều chỉ học từ Train.

In [7]:
X_train_processed=preprocessor.fit_transform(X_train)# fit + transform trên Train
X_val_processed=preprocessor.transform(X_val)# chỉ transform
X_test_processed=preprocessor.transform(X_test)# chỉ transform

print("Kích thước sau khi xử lý (Train):", X_train_processed.shape)

print("Train:", X_train_processed.shape)
print("Val:  ", X_val_processed.shape)
print("Test: ", X_test_processed.shape)

print("Số giá trị thiếu(NAN) còn lại trên Test:", pd.isna(X_test_processed).sum())
print("Số giá trị thiếu(NAN) còn lại trên Val:", pd.isna(X_val_processed).sum())
print("Số giá trị thiếu(NAN) còn lại trên Train:", pd.isna(X_train_processed).sum())

Kích thước sau khi xử lý (Train): (1941, 332)
Train: (1941, 332)
Val:   (648, 332)
Test:  (341, 332)
Số giá trị thiếu(NAN) còn lại trên Test: 0
Số giá trị thiếu(NAN) còn lại trên Val: 0
Số giá trị thiếu(NAN) còn lại trên Train: 0
